# **Domande di analisi**

In [1]:
import pandas as pd
import numpy as np

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
# dati puliti
distribution_center_clean=pd.read_csv("..\\data\\clean_data\\distribution_center_clean.csv")
events_clean=pd.read_csv("..\\data\\clean_data\\events_clean.csv")
inventory_items_clean=pd.read_csv("..\\data\\clean_data\\inventory_items_clean.csv")
order_items_clean=pd.read_csv("..\\data\\clean_data\\order_items_clean.csv")
orders_clean=pd.read_csv("..\\data\\clean_data\\orders_clean.csv")
products_clean=pd.read_csv("..\\data\\clean_data\\products_clean.csv")
users_clean=pd.read_csv("..\\data\\clean_data\\users_clean.csv")

### Website Activity (events)

In [4]:
events_clean.sample(5)

,session_id,user_id,finish,event_type,traffic_source,state,year,month,country
584700,995006b2-bb47-4908-a063-2c9d0c44ba8b,NaN,3,cancel,Adwords,Florida,2025,9,United States
563967,5280440c-b0f4-436d-b40a-ebfd29076964,NaN,1,product,YouTube,Texas,2024,4,United States
516656,3903a148-6ecd-4490-9075-95d6ed171d3c,83338.0,5,purchase,Adwords,Shanghai,2020,3,China
137501,05089e79-6751-4aac-a820-00f917370aaa,NaN,2,product,Adwords,Illinois,2021,4,United States
229995,9ade34c2-ea5d-4761-b706-9ccf351cae35,NaN,3,cart,YouTube,Washington,2021,8,United States


In [5]:
# 1) Quali sono gli eventi finali più frequenti?

In [6]:
events_clean["event_type"].value_counts()
# L'evento finale più frequente è la visualizzazione del prodotto. 

event_type
product     252391
purchase    182539
cart        126263
cancel      126234
Name: count, dtype: int64

In [7]:
# 2) Qual è il tasso di conversione del sito? (purchase/tutte le sessioni)

In [8]:
total_sessions = len(events_clean)
purchase_sessions = (events_clean['event_type'] == 'purchase').sum() 

conversion_rate = purchase_sessions / total_sessions 
conversion_rate_pct = conversion_rate * 100
print(f"Tasso di conversione: {conversion_rate_pct:.2f}%")

Tasso di conversione: 26.55%


In [9]:
# vediamolo per paese
conversion_by_country = events_clean.groupby('country')['event_type'].apply(lambda x: (x == 'purchase').mean())
conversion_by_country=conversion_by_country.sort_values(ascending=False)
conversion_by_country
# Austria ha nettamente la conversione più alta

country
Austria           0.333333
Germany           0.277584
France            0.270565
United Kingdom    0.269328
Japan             0.268468
Brasil            0.268076
Spain             0.267834
Australia         0.266903
China             0.264209
United States     0.263411
Colombia          0.262182
South Korea       0.259353
Belgium           0.251983
Poland            0.246785
Unknown           0.238095
Name: event_type, dtype: float64

In [10]:
events_clean["country"].value_counts()
# Da sottolineare che però l'Austria porta solo 21 sessioni

country
China             231029
United States     153004
Brasil             99304
South Korea        36113
France             32044
United Kingdom     31289
Spain              30224
Germany            28622
Japan              16691
Australia          14923
Belgium             8445
Colombia            4043
Poland              1633
Unknown               42
Austria               21
Name: count, dtype: int64

In [11]:
# è cambiato da anno in anno:
conversion_by_year = events_clean.groupby('year',as_index=False)['event_type'].apply(lambda x: (x == 'purchase').mean())
conversion_by_year=conversion_by_year.sort_values(by='year', ascending=True)
conversion_by_year

,year,event_type
0,2019,0.025161
1,2020,0.078263
2,2021,0.134100
3,2022,0.193642
4,2023,0.260278
5,2024,0.344622
6,2025,0.474276
7,2026,0.688533


In [12]:
"""
KPI molto interessante da utilizzare nella dashboard
Misura con DAX: 
Conversion Rate = 
DIVIDE(
    CALCULATE(COUNTROWS(events_clean), events_clean[event_type] = "purchase"),
    COUNTROWS(events_clean)
)
"""

'\nKPI molto interessante da utilizzare nella dashboard\nMisura con DAX: \nConversion Rate = \nDIVIDE(\n    CALCULATE(COUNTROWS(events_clean), events_clean[event_type] = "purchase"),\n    COUNTROWS(events_clean)\n)\n'

In [13]:
# 3) Quali sorgenti portano più sessioni? Quali più acquisti? Quali hanno il miglior conversion rate?

In [14]:
events_clean["traffic_source"].value_counts()

traffic_source
Email       309632
Adwords     206133
YouTube      68795
Facebook     68475
Organic      34392
Name: count, dtype: int64

In [5]:
# vediamo il conversion rate per traffic_source
conversion_by_traffic_source = events_clean.groupby('traffic_source')['event_type'].apply(lambda x: (x == 'purchase').mean())
conversion_by_traffic_source=conversion_by_traffic_source.sort_values(ascending=False)
conversion_by_traffic_source
# tutte molto allineate

traffic_source
Email       0.266248
Facebook    0.266214
YouTube     0.265310
Adwords     0.265018
Organic     0.261398
Name: event_type, dtype: float64

In [16]:
# 4) Quali combinazioni sono più interessanti?

In [9]:
counts = events_clean.groupby(['traffic_source', 'event_type']).size()
percentages = counts.groupby(level=0).apply(lambda x: x / x.sum())
percentages
# Niente di rilevante. Tutto molto simile

traffic_source  traffic_source  event_type
Adwords         Adwords         cancel        0.183648
                                cart          0.182668
                                product       0.368665
                                purchase      0.265018
Email           Email           cancel        0.182985
                                cart          0.184422
                                product       0.366345
                                purchase      0.266248
Facebook        Facebook        cancel        0.184432
                                cart          0.182037
                                product       0.367317
                                purchase      0.266214
Organic         Organic         cancel        0.187834
                                cart          0.185247
                                product       0.365521
                                purchase      0.261398
YouTube         YouTube         cancel        0.183603
                      

In [18]:
# Un'analisi interessante da fare in Power BI potrebbe essere: osservare il traffic source più utilizzato per il paese selezionato.
# esempio, supponiamo di scegliere il Brasile
brasil=events_clean[events_clean["country"]=="Brasil"]
brasil["traffic_source"].value_counts()

traffic_source
Email       44675
Adwords     29862
Facebook     9968
YouTube      9813
Organic      4986
Name: count, dtype: int64

In [19]:
# Organic indica gli utenti che arrivano sul sito senza che ci sia stata una campagna di marketing pagata o un'azione specifica.
# Adwords  si riferisce al traffico proveniente dalle campagne pubblicitarie a pagamento di Google

### Users

In [20]:
users_clean.sample(5)

,id,first_name,last_name,age,gender,state,city,country,traffic_source,age_group
72707,93614,Tracy,Joseph,18,F,Provence-Alpes-Côte d'Azur,Six-Fours-les-Plages,France,Facebook,Young
73216,97024,Mark,Wood,61,M,Queensland,Jimboomba,Australia,Search,Middle-aged Adult
67459,29896,Mathew,Gutierrez,48,M,Occitanie,Sabran,France,Search,Middle-aged Adult
76970,28683,Robert,Lee,58,M,Seoul,Seoul,South Korea,Display,Middle-aged Adult
23734,14017,Shane,Howell,56,M,England,Ilford,United Kingdom,Search,Middle-aged Adult


In [21]:
# 1) Distribuzione demografica

In [22]:
# - età media
users_clean["age"].mean()

np.float64(40.9496)

In [23]:
users_clean.groupby("country")["age"].mean().sort_values(ascending=False)
# l'età media è molto simile in tutti i country

country
Poland            42.301370
France            41.225223
Belgium           41.135783
Germany           41.119952
Japan             41.065491
United States     41.004052
Austria           41.000000
China             40.946580
Brasil            40.900542
United Kingdom    40.867973
Australia         40.863974
South Korea       40.696558
Spain             40.609529
Colombia          40.285714
Name: age, dtype: float64

In [24]:
# - geografia
users_clean["country"].value_counts()
# La China ha nettamente più utenti.

country
China             34051
United States     22456
Brasil            14579
South Korea        5230
France             4813
United Kingdom     4696
Germany            4185
Spain              3967
Japan              2382
Australia          2154
Belgium            1252
Poland              219
Colombia             14
Austria               2
Name: count, dtype: int64

In [25]:
# -genere
users_clean['gender'].value_counts(normalize=True) * 100

gender
M    50.043
F    49.957
Name: proportion, dtype: float64

In [26]:
gender_by_country = (users_clean.groupby('country')['gender'].value_counts(normalize=True).rename('percentage').reset_index())
gender_by_country
# tutti intorno al 50%, differenza più grande in Colombia

,country,gender,percentage
0,Australia,M,0.517642
1,Australia,F,0.482358
2,Austria,F,0.500000
3,Austria,M,0.500000
4,Belgium,M,0.514377
5,Belgium,F,0.485623
6,Brasil,M,0.505796
7,Brasil,F,0.494204
8,China,F,0.503832
9,China,M,0.496168


### Logistica: inventory_items e distribution_center

In [27]:
inventory_items_clean.sample(5)

,id,product_id,product_distribution_center_id,in_stock,month_year_creation,days_in_stock
142996,123130,21278,1,1,Jun 2025,242
415648,84549,27594,1,1,Jan 2023,1134
53553,88409,7726,9,0,Aug 2025,52
192655,57225,14449,6,1,Nov 2021,1566
55409,395585,7730,10,1,Sep 2020,1979


In [28]:
distribution_center_clean

,id,name,latitude,longitude
0,3,Houston TX,29.7604,-95.3698
1,7,Philadelphia PA,39.9500,-75.1667
2,6,Port Authority of New York/New Jersey NY/NJ,40.6340,-73.7834
3,8,Mobile AL,30.6944,-88.0431
4,1,Memphis TN,35.1174,-89.9711
5,4,Los Angeles CA,34.0500,-118.2500
6,2,Chicago IL,41.8369,-87.6847
7,10,Savannah GA,32.0167,-81.1167
8,9,Charleston SC,32.7833,-79.9333
9,5,New Orleans LA,29.9500,-90.0667


In [29]:
# 1) Stock per distribution center

# Calcolo articoli in stock (=1)
in_stock = (
    inventory_items_clean
    .assign(in_stock_flag = inventory_items_clean["in_stock"] == 1)
    .groupby("product_distribution_center_id")["in_stock_flag"]
    .sum()
    .reset_index()
    .rename(columns={"in_stock_flag": "items_in_stock"})
)

# Calcolo articoli esauriti (=0)
out_of_stock = (
    inventory_items_clean
    .assign(out_stock_flag = inventory_items_clean["in_stock"] == 0)
    .groupby("product_distribution_center_id")["out_stock_flag"]
    .sum()
    .reset_index()
    .rename(columns={"out_stock_flag": "items_out_of_stock"})
)

# Merge dei due risultati
dist_stock = (
    in_stock
    .merge(out_of_stock, on="product_distribution_center_id")
    .merge(distribution_center_clean, left_on="product_distribution_center_id", right_on="id")
)

# Selezione colonne finali
dist_stock = dist_stock[["id", "name", "items_in_stock", "items_out_of_stock"]]

dist_stock


,id,name,items_in_stock,items_out_of_stock
0,1,Memphis TN,40999,24005
1,2,Chicago IL,40636,23977
2,3,Houston TX,38813,22848
3,4,Los Angeles CA,29735,17477
4,5,New Orleans LA,22076,12952
5,6,Port Authority of New York/New Jersey NY/NJ,27699,16211
6,7,Philadelphia PA,28855,16941
7,8,Mobile AL,31043,18304
8,9,Charleston SC,28476,16831
9,10,Savannah GA,20087,11858


In [30]:
# 2) media dei giorni che un prodotto rimane in stock prima di essere venduto, a quelli non venduti è stato calcolato come la differenza tra la data di oggi (19/02/2026) e la data di creazione
inventory_items_clean.groupby("product_distribution_center_id", as_index=False).agg({"days_in_stock": "mean"}).sort_values("days_in_stock", ascending=False)

,product_distribution_center_id,days_in_stock
2,3,721.623376
0,1,721.071811
6,7,720.620884
8,9,720.113316
9,10,719.156613
5,6,717.725188
7,8,717.589539
3,4,716.569262
1,2,715.877532
4,5,713.848293


In [31]:
# 3) Quali categorie di prodotti sono più presenti in ogni centro?

In [32]:
inv_prod_cat=pd.merge(left=inventory_items_clean, right=products_clean, left_on="product_id", right_on="id", how="left")
inv_prod_cat=inv_prod_cat[["id_x","product_id","product_distribution_center_id","in_stock","category"]]
inv_prod_cat

,id_x,product_id,product_distribution_center_id,in_stock,category
0,174571,13844,7,0,Accessories
1,174572,13844,7,1,Accessories
2,216419,13844,7,0,Accessories
3,216420,13844,7,1,Accessories
4,216421,13844,7,1,Accessories
...,...,...,...,...,...
489818,332010,25590,3,1,Underwear
489819,467577,25590,3,0,Underwear
489820,467578,25590,3,1,Underwear
489821,467579,25590,3,1,Underwear


In [33]:
best_cat_dist = inv_prod_cat.groupby(["product_distribution_center_id", "category"], as_index=False)["in_stock"].sum()
best_cat_dist["rank"] = best_cat_dist.groupby("product_distribution_center_id")["in_stock"].rank(method="dense", ascending=False)
best_cat_dist = best_cat_dist.sort_values(["product_distribution_center_id", "rank"])
best_cat_dist = best_cat_dist[best_cat_dist["rank"] == 1]
best_cat_dist

,product_distribution_center_id,category,in_stock,rank
24,1,Tops & Tees,3121,1.0
32,2,Intimates,4823,1.0
58,3,Intimates,3676,1.0
101,4,Swim,4029,1.0
111,5,Jeans,2009,1.0
153,6,Tops & Tees,2246,1.0
161,7,Jeans,3583,1.0
187,8,Jeans,3761,1.0
210,9,Fashion Hoodies & Sweatshirts,2886,1.0
255,10,Underwear,3050,1.0


### Analisi sui prodotti (products, order_items e users)

In [34]:
products_clean.sample(5)

,id,cost,category,brand,retail_price,department,distribution_center_id,ideal_profit
16380,16456,11.10,Tops & Tees,FEA,20.00,Men,6,8.90
4184,15336,18.80,Plus,Bali,38.52,Women,2,19.72
14958,24846,12.87,Socks,KENTWOOL,19.99,Men,5,7.12
14037,14512,20.65,Maternity,Bravado! Designs,49.99,Women,4,29.34
8583,11761,12.50,Intimates,ADVADRI,23.99,Women,3,11.49


In [35]:
users_clean.sample(5)

,id,first_name,last_name,age,gender,state,city,country,traffic_source,age_group
61949,68728,John,Winters,39,M,New South Wales,Sydney,Australia,Search,Adult
44245,3612,Judy,Oliver,24,F,Hiroshima,Hiroshima city,Japan,Search,Young
969,33777,Nicholas,Sandoval,60,M,Amazonas,Manaus,Brasil,Search,Middle-aged Adult
14025,35150,Kathleen,Davis,45,F,Canarias,San Cristóbal de La Laguna,Spain,Organic,Middle-aged Adult
82771,71493,Margaret,Berry,68,F,Shanghai,Xianyang,China,Search,Senior


In [36]:
order_items_clean.sample(5)

,id,order_id,user_id,product_id,inventory_item_id,status,sale_price,cost,profit_per_product
43329,19298,13429,10805,18471,51925,Complete,23.99,10.27,13.72
150581,27941,19399,15589,20821,75189,Processing,90.00,44.46,45.54
157081,96449,66738,53402,19196,260285,Complete,104.50,52.98,51.52
83097,60073,41578,33408,26544,162003,Returned,36.99,15.54,21.45
150122,25708,17842,14283,23756,69181,Processing,89.99,43.20,46.79


In [37]:
# Creiamo una tabella unica che ci aiuterà nell'analisi 
all=pd.merge(left=order_items_clean, right=products_clean, how='left', left_on='product_id', right_on='id')
all=all[["id_x","order_id","user_id","product_id","status","sale_price","cost_x","profit_per_product","category","brand","department"]]
all.rename(columns={"id_x": "id", "cost_x": "cost"}, inplace=True)
all.sample(5)

,id,order_id,user_id,product_id,status,sale_price,cost,profit_per_product,category,brand,department
79964,54885,38063,30647,23600,Shipped,35.90,17.63,18.27,Shorts,Metal Mulisha,Men
26994,14316,9945,7952,8976,Processing,16.99,6.24,10.75,Socks & Hosiery,Anne Klein,Women
107992,134700,92924,74345,3800,Shipped,49.50,21.14,28.36,Dresses,Evan Picone,Women
166291,14914,10381,8299,20218,Complete,139.99,51.80,88.19,Suits & Sport Coats,Tasso Elba,Men
146196,74604,51629,41313,18796,Processing,84.06,36.31,47.75,Active,2XU,Men


In [38]:
all2=pd.merge(all, users_clean, left_on='user_id', right_on='id', how='left')
all2=all2[["id_x", "order_id", "user_id", "product_id", "status", "sale_price", "cost", "profit_per_product", "category", "brand", "department", "gender", "country", "age_group"]]
all2.rename(columns={"id_x": "id"}, inplace=True)
all2.sample(5)

,id,order_id,user_id,product_id,status,sale_price,cost,profit_per_product,category,brand,department,gender,country,age_group
22767,40405,28061,22557,8912,Shipped,15.00,6.72,8.28,Socks & Hosiery,Gold Toe,Women,F,China,Adult
322,114416,79011,63224,9013,Complete,3.39,1.43,1.96,Socks & Hosiery,Chasse,Women,F,United States,Middle-aged Adult
32171,95173,65829,52695,25199,Returned,19.39,12.45,6.94,Socks,Browning,Men,M,China,Adult
146935,112706,77837,62248,19713,Shipped,85.00,42.84,42.16,Sweaters,Tommy Bahama,Men,M,United States,Young
28836,107425,74178,59350,26826,Cancelled,17.99,7.34,10.65,Sleep & Lounge,Acura,Men,M,China,Middle-aged Adult


In [39]:
# 1) Analizziamo le categorie

In [40]:
# teniamo solo gli ordini in processo, spediti, completati e non restituiti o cancellati
all_no_ret_canc = all2[(all2["status"] != "Returned") & (all2["status"] != "Cancelled")]

In [41]:
# categorie più vendute
cat_most_sal=all_no_ret_canc.groupby("category",as_index=False)["product_id"].size().sort_values(by="size",ascending=False)
cat_most_sal

,category,size
6,Intimates,10082
7,Jeans,9714
24,Tops & Tees,9064
5,Fashion Hoodies & Sweatshirts,8933
15,Shorts,8447
22,Sweaters,8433
23,Swim,8415
17,Sleep & Lounge,8395
0,Accessories,7412
1,Active,6792


In [42]:
# saranno le stesse 3 ad avere il miglior profitto?
all_no_ret_canc.groupby("category", as_index=False)["profit_per_product"].sum().sort_values("profit_per_product", ascending=False).head(3)

,category,profit_per_product
11,Outerwear & Coats,547393.18
7,Jeans,441072.89
22,Sweaters,330524.90


In [ ]:
"""
Potremmo osservare tantissime cose. Un'idea è che in Power BI andiamo a selezionare un country 
e per esso visualizziamo le categorie più profittevoli, le più vendute, con una suddivisione anche per gruppo d'età.
"""

"\nPotremmo osservare tantissime cose. La mia idea è che in Power BI andiamo a selezionare un country \ne per esso visualizziamo le categorie più profittevoli, le più vendute, con una suddivisione anche per gruppo d'età.\n"

In [44]:
# 2) focus sui resi

In [45]:
all_ret=all2[(all2["status"] == "Returned")]

In [46]:
cat_most_ret=all_ret.groupby(["category"],as_index=False)["product_id"].size().sort_values(by="size", ascending=False)
cat_most_ret

,category,size
6,Intimates,1317
7,Jeans,1282
24,Tops & Tees,1247
5,Fashion Hoodies & Sweatshirts,1183
15,Shorts,1145
17,Sleep & Lounge,1141
23,Swim,1131
22,Sweaters,1114
0,Accessories,978
1,Active,938


In [47]:
# Calcoliamo la percentuale di prodotti resi rispetto al totale

In [48]:
perc_ret=pd.merge(cat_most_sal, cat_most_ret, on='category', how='inner')
perc_ret.rename(columns={'size_x':'saled', 'size_y':'returned'}, inplace=True)
perc_ret["total"] = perc_ret["saled"] + perc_ret["returned"]
perc_ret["perc_returned"] = round((perc_ret["returned"] / perc_ret["total"])*100,2)
perc_ret=perc_ret[["category", "perc_returned"]]
perc_ret=perc_ret.sort_values(by='perc_returned', ascending=False)
perc_ret
# la percentuale di reso può essere una misura interessante da utilizzare in Power BI

,category,perc_returned
25,Clothing Sets,13.56
21,Blazers & Jackets,13.12
23,Suits,12.78
17,Plus,12.69
18,Socks & Hosiery,12.52
24,Jumpsuits & Rompers,12.42
20,Leggings,12.36
9,Active,12.13
10,Outerwear & Coats,12.11
2,Tops & Tees,12.09


In [49]:
""" Misura percentuale resi in DAX:
Returned Items = 
CALCULATE(
    COUNTROWS(order_items_clean),
    order_items_clean[status] = "Returned"
)

Sold Items =
CALCULATE(
    COUNTROWS(order_items_clean),
    order_items_clean[status] <> "Cancelled"
)

Return Rate = 
DIVIDE([Returned Items], [Sold Items])
"""

' Misura percentuale resi in DAX:\nReturned Items = \nCALCULATE(\n    COUNTROWS(order_items_clean),\n    order_items_clean[status] = "Returned"\n)\n\nSold Items =\nCALCULATE(\n    COUNTROWS(order_items_clean),\n    order_items_clean[status] <> "Cancelled"\n)\n\nReturn Rate = \nDIVIDE([Returned Items], [Sold Items])\n'

### Analisi sugli ordini (orders, users)

In [50]:
orders_clean.sample(5)

,order_id,user_id,status,num_of_item,month_year_creation,days_for_shipping,days_for_delivery,year,month,sale_price,cost,profit
68785,84110,67304,Cancelled,1,Nov 2025,NaN,NaN,2025,11,15.95,9.12,0.00
25107,1390,1117,Processing,1,Apr 2021,NaN,NaN,2021,4,42.00,23.10,18.90
83593,94272,75423,Complete,2,Jan 2026,0.0,4.0,2026,1,144.45,62.00,82.45
8390,112283,89861,Cancelled,2,Mar 2021,NaN,NaN,2021,3,196.28,80.25,0.00
46263,16379,13119,Shipped,1,Nov 2025,2.0,NaN,2025,11,24.99,11.47,13.52


In [51]:
users_clean.sample(5)

,id,first_name,last_name,age,gender,state,city,country,traffic_source,age_group
40419,43110,Neil,Gilmore,24,M,Hebei,Handan,China,Search,Young
1745,15982,Jacob,Thompson,49,M,Andalucía,Málaga,Spain,Email,Middle-aged Adult
97710,69906,Gregg,Stone,37,M,Zhejiang,Tianjin,China,Facebook,Adult
68775,39194,Charles,Hammond,15,M,Paraná,Londrina,Brasil,Search,Young
20483,14148,Richard,Pena,19,M,England,Bristol,United Kingdom,Organic,Young


In [52]:
# Creare tabella unica per facilitare le analisi
ord_use=pd.merge(orders_clean, users_clean, left_on='user_id', right_on='id', how='left')
ord_use=ord_use[["order_id", "user_id","status","num_of_item","month_year_creation","days_for_shipping","days_for_delivery",
"year","month","sale_price","cost","profit","age","gender","country","age_group"]]
ord_use.sample(5)

,order_id,user_id,status,num_of_item,month_year_creation,days_for_shipping,days_for_delivery,year,month,sale_price,cost,profit,age,gender,country,age_group
71263,117030,93677,Cancelled,1,Oct 2020,NaN,NaN,2020,10,225.00,94.27,0.0,62,M,France,Middle-aged Adult
123252,113789,91048,Shipped,3,Aug 2025,0.0,NaN,2025,8,331.49,157.49,174.0,33,M,United Kingdom,Adult
100812,16457,13182,Returned,2,Apr 2024,0.0,2.0,2024,4,46.94,19.72,0.0,58,M,United States,Middle-aged Adult
64237,22319,17924,Cancelled,1,Jun 2024,NaN,NaN,2024,6,30.00,17.37,0.0,53,M,China,Middle-aged Adult
38342,17421,13936,Returned,2,Jul 2023,1.0,0.0,2023,7,199.99,101.64,0.0,47,F,Spain,Middle-aged Adult


In [53]:
# 1) paesi con più ordini e con più profitto

In [54]:
ord_use.groupby("country", as_index=False)["order_id"].size().sort_values(by="size", ascending=False)

,country,size
4,China,42704
13,United States,28022
3,Brasil,18046
10,South Korea,6461
6,France,6014
12,United Kingdom,5874
7,Germany,5360
11,Spain,4922
8,Japan,3076
0,Australia,2669


In [55]:
ord_use.groupby("country", as_index=False)["profit"].sum().sort_values(by="profit", ascending=False)

,country,profit
4,China,1442929.35
13,United States,946597.09
3,Brasil,592450.48
10,South Korea,219388.09
12,United Kingdom,203871.85
6,France,199724.04
7,Germany,180809.12
11,Spain,164628.17
8,Japan,99856.11
0,Australia,89167.92


In [56]:
ord_use.groupby("country", as_index=False)["profit"].mean().sort_values(by="profit", ascending=False)

,country,profit
12,United Kingdom,34.707499
9,Poland,34.176245
10,South Korea,33.955748
4,China,33.789091
13,United States,33.780497
7,Germany,33.733045
2,Belgium,33.489922
11,Spain,33.447414
0,Australia,33.408737
6,France,33.209850


In [57]:
# Colombia e Austria sono le peggiori per numero di ordini, profitto totale e media del profitto per ogni ordine

In [58]:
# 2) Come sta andando nel corso degli anni?

In [59]:
ord_use.groupby("year", as_index=False)["profit"].sum().sort_values(by="year", ascending=True)
# Molto in crescita, il 2026 ha solo i primi due mesi.
# Sarà interessante vederlo su PowerBI, confrontando i vari paesi.

,year,profit
0,2019,41849.21
1,2020,134726.74
2,2021,250746.86
3,2022,385549.14
4,2023,592632.63
5,2024,860071.09
6,2025,1473116.44
7,2026,462389.02


In [60]:
ord_use["month_year_creation"].value_counts()

month_year_creation
Feb 2026    7128
Jan 2026    6661
Dec 2025    5437
Nov 2025    4734
Oct 2025    4393
            ... 
May 2019      83
Apr 2019      71
Mar 2019      51
Feb 2019      22
Jan 2019      16
Name: count, Length: 86, dtype: int64

In [61]:
# cambia il modo di acquistare in base all'età?

In [62]:
ord_use.groupby("age_group", as_index=False)["sale_price"].mean().sort_values(by="sale_price", ascending=False)
# Anche qui non sembra incidere più di tanto a livello generale, magari in qualche paese c'è una differenza più marcata. 

,age_group,sale_price
2,Senior,87.003372
1,Middle-aged Adult,86.503713
0,Adult,86.454370
3,Young,86.197559


In [63]:
# 3) stagionalità
# Supponiamo di concentraric sul brasile: in quali mesi spendono di più? in quali fanno più acquisti?

In [64]:
ord_use[ord_use['country']=='Brasil'].groupby("month", as_index=False)["sale_price"].mean().sort_values(by="sale_price", ascending=False)

,month,sale_price
3,4,91.156303
11,12,88.378047
2,3,87.420597
0,1,86.980224
9,10,86.381895
6,7,86.267607
7,8,85.965105
10,11,84.943306
4,5,83.906693
8,9,83.617734


In [65]:
ord_use[ord_use['country']=='Brasil'].groupby("month", as_index=False)["order_id"].size().sort_values(by="month", ascending=True)
# Nel periodo tra dicembre e febbraio i brasiliani tendono a fare più acquisti.

,month,size
0,1,2055
1,2,1987
2,3,1105
3,4,1147
4,5,1261
5,6,1222
6,7,1425
7,8,1379
8,9,1390
9,10,1562


In [66]:
# negli Stati Uniti è la stessa cosa? sembra di si
ord_use[ord_use['country']=='United States'].groupby("month", as_index=False)["order_id"].size().sort_values(by="month", ascending=True)

,month,size
0,1,3113
1,2,3063
2,3,1883
3,4,1816
4,5,2001
5,6,1901
6,7,2104
7,8,2190
8,9,2229
9,10,2382


In [67]:
# è una tendenza generale del sito?

In [68]:
ord_use.groupby("month", as_index=False)["order_id"].size().sort_values(by="month", ascending=True)
# Sembra proprio di sì. I periodi invernali sono quelli che portano a più acquisti.

,month,size
0,1,13950
1,2,13938
2,3,7858
3,4,8121
4,5,8693
5,6,8598
6,7,9293
7,8,9803
8,9,9953
9,10,10863


In [69]:
ord_use.groupby("user_id", as_index=False)["order_id"].size().sort_values(by="size", ascending=False).head(10)
# Il massimo numero di ordini per un singolo utente è 4. Sono in tanti con 4 ordini. Vediamo se otteniamo informazioni migliori tenendo conto della spesa.

,user_id,size
29798,37288,4
74115,92684,4
42137,52676,4
26420,33093,4
74105,92671,4
18742,23542,4
26427,33101,4
14354,18023,4
9987,12534,4
48567,60740,4


In [70]:
# 10 clienti che hanno speso di più
top10_users=ord_use.groupby("user_id", as_index=False)["sale_price"].sum().sort_values(by="sale_price", ascending=False).head(10)

In [71]:
top10_users_merged = top10_users.merge(users_clean, left_on="user_id", right_on="id", how='inner')
top10_users_merged

,user_id,sale_price,id,first_name,last_name,age,gender,state,city,country,traffic_source,age_group
0,58819,1817.70,58819,Leonard,Smith,45,M,Guangdong,Wuhan,China,Search,Middle-aged Adult
1,56589,1707.97,56589,Richard,Smith,16,M,Cataluña,Terrassa,Spain,Search,Young
2,34715,1663.45,34715,Joseph,Davis,26,M,Beijing,Weihai,China,Organic,Adult
3,32484,1586.62,32484,Lynn,Mathews,42,F,Shanghai,Qinhuangdao,China,Search,Adult
4,12322,1558.28,12322,Anthony,King,67,M,Berlin,Berlin,Germany,Email,Senior
5,71039,1497.87,71039,Carly,Villa,62,F,Henan,Fushun,China,Search,Middle-aged Adult
6,26088,1491.98,26088,Kerri,Freeman,12,F,Sichuan,Wenzhou,China,Organic,Young
7,38791,1451.62,38791,Frances,Lucas,13,F,Shanghai,Nanchong,China,Facebook,Young
8,18010,1392.18,18010,Diane,Keller,51,F,Oregon,Newberg,United States,Organic,Middle-aged Adult
9,83570,1391.49,83570,Jessica,Li,70,F,Minas Gerais,Uberlândia,Brasil,Search,Senior


In [72]:
top10_users_merged=top10_users_merged[['country', 'age', 'gender', 'sale_price']]
top10_users_merged.columns =['country', 'age', 'gender', 'money_spend']

In [73]:
top10_users_merged
# 6 cinesi, 2 giovanissime

,country,age,gender,money_spend
0,China,45,M,1817.70
1,Spain,16,M,1707.97
2,China,26,M,1663.45
3,China,42,F,1586.62
4,Germany,67,M,1558.28
5,China,62,F,1497.87
6,China,12,F,1491.98
7,China,13,F,1451.62
8,United States,51,F,1392.18
9,Brasil,70,F,1391.49
